# Exploring Relationships Between Variables Using Scatter Plots

**Milestone: Scatter Plot Relationship Analysis**  
**Project: Trivin Insight Engine**  
**Date: March 2026**

---

## Learning Objectives

By completing this notebook, you will be able to:

1. Create scatter plots for two numeric variables
2. Interpret positive, negative, or no clear relationship
3. Distinguish linear and non-linear visual patterns
4. Spot clusters and outliers
5. Use scatter plots to guide deeper EDA decisions

This lesson is descriptive EDA only. No regression or modeling is required.

## Setup: Import Libraries and Load Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries imported successfully')

In [ ]:
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

csv_path = project_root / 'data' / 'raw' / 'employee_survey_2026_Q1.csv'
figure_dir = project_root / 'outputs' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path, parse_dates=['survey_date'])
print(f'Dataset loaded from: {csv_path}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print('\nNumeric columns:')
print(numeric_columns)

df.head()

## Part 1: Understanding Scatter Plots

A scatter plot places one numeric variable on the x-axis and another on the y-axis.
Each point represents one observation.

Use scatter plots to inspect:

- direction: positive, negative, or unclear
- strength: tight or diffuse point cloud
- shape: linear or non-linear
- clusters: grouped points
- outliers: isolated points

Scatter plots show relationships, not sequences over time.

In [ ]:
x_col = 'work_life_balance'
y_col = 'satisfaction_score'

fig, ax = plt.subplots()
ax.scatter(df[x_col], df[y_col], color='steelblue', alpha=0.8, s=65, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Work-Life Balance', fontsize=12)
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_title('Satisfaction vs Work-Life Balance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_scatter_satisfaction_vs_work_life_balance.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/figures/notebook_scatter_satisfaction_vs_work_life_balance.png')

### Interpretation Prompts

- Do points generally move upward from left to right?
- Is the cloud tight (stronger) or spread out (weaker)?
- Do you notice any isolated points?

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['management_support'], df['satisfaction_score'], color='darkorange', alpha=0.8, s=65, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Management Support', fontsize=12)
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_title('Satisfaction vs Management Support', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_scatter_satisfaction_vs_management_support.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/figures/notebook_scatter_satisfaction_vs_management_support.png')

## Part 2: Linear vs Non-Linear Pattern Awareness

Many real-world relationships are not perfectly linear.
When reading a scatter plot, avoid forcing a straight-line interpretation if the cloud appears curved or segmented.

For this dataset, focus on visual structure:

- roughly straight trend: linear tendency
- curved pattern: non-linear tendency
- no path: no obvious pattern

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
departments = sorted(df['department'].dropna().unique())
colors = plt.cm.get_cmap('tab10', len(departments))

for idx, dept in enumerate(departments):
    subset = df[df['department'] == dept]
    ax.scatter(subset['career_growth'], subset['satisfaction_score'],
               color=colors(idx), s=70, alpha=0.8, edgecolor='black', linewidth=0.4, label=dept)

ax.set_xlabel('Career Growth', fontsize=12)
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_title('Career Growth vs Satisfaction (Colored by Department)', fontsize=14, fontweight='bold')
ax.legend(title='Department', ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_scatter_department_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/figures/notebook_scatter_department_clusters.png')

### Cluster Questions

- Do some departments appear concentrated in different regions?
- Are there overlaps or distinct groupings?
- What business questions could this pattern suggest?

In [ ]:
x_col = 'work_life_balance'
y_col = 'satisfaction_score'

x_q1, x_q3 = df[x_col].quantile(0.25), df[x_col].quantile(0.75)
y_q1, y_q3 = df[y_col].quantile(0.25), df[y_col].quantile(0.75)

x_iqr = x_q3 - x_q1
y_iqr = y_q3 - y_q1

x_low, x_high = x_q1 - 1.5 * x_iqr, x_q3 + 1.5 * x_iqr
y_low, y_high = y_q1 - 1.5 * y_iqr, y_q3 + 1.5 * y_iqr

outliers = df[(df[x_col] < x_low) | (df[x_col] > x_high) | (df[y_col] < y_low) | (df[y_col] > y_high)]
typical = df.drop(outliers.index)

fig, ax = plt.subplots()
ax.scatter(typical[x_col], typical[y_col], color='slategray', alpha=0.7, s=60, edgecolor='black', linewidth=0.4, label='Typical')
ax.scatter(outliers[x_col], outliers[y_col], color='crimson', alpha=0.95, s=95, edgecolor='black', linewidth=0.6, label='Potential outlier')

for _, row in outliers.iterrows():
    ax.annotate(str(row['employee_id']), (row[x_col], row[y_col]), textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Work-Life Balance', fontsize=12)
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_title('Outlier Highlighting in Scatter Plot', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_scatter_outlier_highlight.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Potential outliers found: {len(outliers)}')
print('Saved: outputs/figures/notebook_scatter_outlier_highlight.png')
outliers[['employee_id', 'department', x_col, y_col]].head()

## Part 3: Wrap-Up and Reflection

Write short answers below (in your own words):

1. Which variable pair showed the clearest relationship direction?
2. Did you observe any linear or non-linear pattern?
3. Did you identify any clusters?
4. Which points looked like potential outliers?
5. Why are scatter plots useful before deeper analysis?

### Video Walkthrough Checklist (~2 minutes)

- Show one scatter plot creation
- Explain relationship direction
- Mention trends, clusters, and outliers
- Explain why scatter plots are useful in EDA

Remember: correlation patterns do not prove causation.